In [1]:
import os, re, gc, random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA available: True
GPU: Tesla T4


In [2]:
MODEL_ID = "google/gemma-3-4b-it"

DATA_DIR = "/kaggle/input/datasets/taufiqzahantushar/checksent-bn/CheckSent-BN"
TRAIN_PATH = os.path.join(DATA_DIR, "train_data.tsv")
TEST_PATH = os.path.join(DATA_DIR, "test_data.tsv")

LABELS = ["NEGATIVE", "NEUTRAL", "POSITIVE"]

SEED = 42
K_PER_CLASS = 2            # examples per class (2 -> 6 shots total; try 3 for 9)
MAX_NEW_TOKENS = 8
BATCH_SIZE = 8             # lower to 4 if you get CUDA OOM

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [3]:
train_df = pd.read_csv(TRAIN_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")

print("Train:", train_df.shape, "| Test:", test_df.shape)
print("\nTest distribution:")
print(test_df["Sentiment"].value_counts())
print("\nMissing headlines:", train_df["Headline"].isna().sum(), test_df["Headline"].isna().sum())

Train: (9254, 6) | Test: (2314, 6)

Test distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64

Missing headlines: 0 0


In [4]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("tushartoken")
print("HF token loaded.")

HF token loaded.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "left"          # required for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.eval()
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded on: cuda:0


## Select few-shot examples
Balanced per class, fixed seed, shuffled order (avoids recency bias).

In [6]:
examples = []
for label in LABELS:
    pool = train_df[train_df["Sentiment"] == label]
    pool = pool[pool["Headline"].str.len().between(30, 120)]
    picks = pool.sample(n=K_PER_CLASS, random_state=SEED)
    for h in picks["Headline"]:
        examples.append({"headline": h, "label": label})

random.Random(SEED).shuffle(examples)
few_shot_examples = examples

print("Few-shot examples:\n")
for e in few_shot_examples:
    print(f"[{e['label']}] {e['headline']}")

Few-shot examples:

[NEUTRAL] রবিবার সামনে নর্থইস্ট, কোন স্ট্র্যাটেজিতে জয়ের ছক কষছেন মোলিনা?
[NEGATIVE] ‘ফেনজলে’র তাণ্ডবে বেসামাল বিমান, পাইলটের তৎপরতায় বাঁচল শতাধিক প্রাণ, ভাইরাল ভিডিও
[NEUTRAL] রাজ্যসভার প্রার্থী শ্রমিক নেতা ঋতব্রত বন্দ্যোপাধ্যায়, ঘোষণা তৃণমূলের
[POSITIVE] ‘ভারতীয় ফুটবলে বাঙালির দাপট আবার ফিরবেই’
[NEGATIVE] একের পর এক ইজরায়েলি বোমার আঘাতে কার্যত ধ্বংসস্তূপে পরিণত হয়েছে...
[POSITIVE] পুজোয় ‘দ্য ভোজ কোম্পানি’র লোভনীয় মেনু লঞ্চ, হাজির টলি তারকা ঐশ্বর্য সেন


## Prompt (Gemma chat template) and predictor

In [7]:
INSTRUCTION = """You are performing Bengali news-headline sentiment classification.

Labels:
NEGATIVE = the headline expresses a negative sentiment.
NEUTRAL  = the headline expresses a neutral or factual sentiment.
POSITIVE = the headline expresses a positive sentiment.

Reply with ONLY one word: NEGATIVE, NEUTRAL, or POSITIVE. No explanation, no code block."""


def build_few_shot_messages(headline):
    messages = []
    for i, ex in enumerate(few_shot_examples):
        user_text = f"Headline: {ex['headline']}"
        if i == 0:   # Gemma has no system role -> put rules in the first user turn
            user_text = INSTRUCTION + "\n\n" + user_text
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": ex["label"]})
    messages.append({"role": "user", "content": f"Headline: {headline}"})
    return messages


def build_few_shot_text(headline):
    return tokenizer.apply_chat_template(
        build_few_shot_messages(headline),
        tokenize=False,
        add_generation_prompt=True,
    )


LABEL_RE = re.compile(r"\b(NEGATIVE|NEUTRAL|POSITIVE)\b", re.IGNORECASE)

def parse_sentiment(output):
    m = LABEL_RE.search(output or "")
    return m.group(1).upper() if m else None


@torch.no_grad()
def predict_few_shot_batch(headlines):
    texts = [build_few_shot_text(h) for h in headlines]
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,      # template already includes <bos>
    ).to(model.device)

    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    raws = [r.strip() for r in tokenizer.batch_decode(gen, skip_special_tokens=True)]
    return [parse_sentiment(r) for r in raws], raws


print("Example prompt:\n")
print(build_few_shot_text(test_df.iloc[3]["Headline"]))

Example prompt:

<bos><start_of_turn>user
You are performing Bengali news-headline sentiment classification.

Labels:
NEGATIVE = the headline expresses a negative sentiment.
NEUTRAL  = the headline expresses a neutral or factual sentiment.
POSITIVE = the headline expresses a positive sentiment.

Reply with ONLY one word: NEGATIVE, NEUTRAL, or POSITIVE. No explanation, no code block.

Headline: রবিবার সামনে নর্থইস্ট, কোন স্ট্র্যাটেজিতে জয়ের ছক কষছেন মোলিনা?<end_of_turn>
<start_of_turn>model
NEUTRAL<end_of_turn>
<start_of_turn>user
Headline: ‘ফেনজলে’র তাণ্ডবে বেসামাল বিমান, পাইলটের তৎপরতায় বাঁচল শতাধিক প্রাণ, ভাইরাল ভিডিও<end_of_turn>
<start_of_turn>model
NEGATIVE<end_of_turn>
<start_of_turn>user
Headline: রাজ্যসভার প্রার্থী শ্রমিক নেতা ঋতব্রত বন্দ্যোপাধ্যায়, ঘোষণা তৃণমূলের<end_of_turn>
<start_of_turn>model
NEUTRAL<end_of_turn>
<start_of_turn>user
Headline: ‘ভারতীয় ফুটবলে বাঙালির দাপট আবার ফিরবেই’<end_of_turn>
<start_of_turn>model
POSITIVE<end_of_turn>
<start_of_turn>user
Headline: একের 

## Sanity check (first run this before the full evaluation)

In [8]:
idxs = [0, 1, 2, 3, 5, 6, 20]
preds, raws = predict_few_shot_batch(test_df.iloc[idxs]["Headline"].tolist())

for i, p, r in zip(idxs, preds, raws):
    actual = test_df.iloc[i]["Sentiment"]
    print(f"Sample {i:>3} | raw: {r!r:<14} | pred: {p} | actual: {actual} | {'OK' if p == actual else 'WRONG'}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sample   0 | raw: 'NEGATIVE'     | pred: NEGATIVE | actual: NEGATIVE | OK
Sample   1 | raw: 'NEGATIVE'     | pred: NEGATIVE | actual: NEGATIVE | OK
Sample   2 | raw: 'NEUTRAL'      | pred: NEUTRAL | actual: NEUTRAL | OK
Sample   3 | raw: 'NEGATIVE'     | pred: NEGATIVE | actual: NEGATIVE | OK
Sample   5 | raw: 'NEGATIVE'     | pred: NEGATIVE | actual: NEGATIVE | OK
Sample   6 | raw: 'POSITIVE'     | pred: POSITIVE | actual: POSITIVE | OK
Sample  20 | raw: 'POSITIVE'     | pred: POSITIVE | actual: POSITIVE | OK


## Full few-shot evaluation

In [9]:
gc.collect(); torch.cuda.empty_cache()

few_shot_predictions, few_shot_raw_outputs = [], []
headlines = test_df["Headline"].tolist()

for start in tqdm(range(0, len(headlines), BATCH_SIZE), desc="Few-shot"):
    p, r = predict_few_shot_batch(headlines[start:start + BATCH_SIZE])
    few_shot_predictions.extend(x if x is not None else "UNKNOWN" for x in p)
    few_shot_raw_outputs.extend(r)

print("\nTotal predictions  :", len(few_shot_predictions))
print("Unknown predictions:", few_shot_predictions.count("UNKNOWN"))

Few-shot:   0%|          | 0/290 [00:00<?, ?it/s]


Total predictions  : 2314
Unknown predictions: 0


In [10]:
results_df = test_df[["ID", "Headline", "Sentiment"]].copy()
results_df["few_shot_pred"] = few_shot_predictions
results_df["few_shot_raw"] = few_shot_raw_outputs
results_df.to_csv("few_shot_predictions.csv", index=False, encoding="utf-8-sig")
print("Saved: few_shot_predictions.csv")

Saved: few_shot_predictions.csv


## Metrics

In [11]:
y_true = test_df["Sentiment"].tolist()
y_pred = few_shot_predictions

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=LABELS, average="macro", zero_division=0
)

print("=" * 60)
print("FEW-SHOT EVALUATION RESULTS")
print("=" * 60)
print(f"Total Samples       : {len(y_true)}")
print(f"Unknown Predictions : {y_pred.count('UNKNOWN')}")
print(f"\nAccuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}  (macro)")
print(f"Recall    : {recall:.4f}  (macro)")
print(f"F1-score  : {f1:.4f}  (macro)")

print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)
print(classification_report(y_true, y_pred, labels=LABELS, digits=4, zero_division=0))

print("Confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=LABELS),
    index=[f"true_{l}" for l in LABELS],
    columns=[f"pred_{l}" for l in LABELS],
))

FEW-SHOT EVALUATION RESULTS
Total Samples       : 2314
Unknown Predictions : 0

Accuracy  : 0.7882
Precision : 0.7760  (macro)
Recall    : 0.7470  (macro)
F1-score  : 0.7525  (macro)

CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.7947    0.9688    0.8732      1123
     NEUTRAL     0.8127    0.5891    0.6831       825
    POSITIVE     0.7205    0.6831    0.7013       366

    accuracy                         0.7882      2314
   macro avg     0.7760    0.7470    0.7525      2314
weighted avg     0.7894    0.7882    0.7782      2314

Confusion matrix (rows = actual, cols = predicted):
               pred_NEGATIVE  pred_NEUTRAL  pred_POSITIVE
true_NEGATIVE           1088            28              7
true_NEUTRAL             249           486             90
true_POSITIVE             32            84            250


In [12]:
# Inspect any remaining UNKNOWN outputs
unk = [i for i, p in enumerate(few_shot_predictions) if p == "UNKNOWN"]
print("UNKNOWN count:", len(unk))
for i in unk[:10]:
    print(i, "|", test_df.iloc[i]["Headline"], "|", repr(few_shot_raw_outputs[i]))

UNKNOWN count: 0
